# 18 günlük ayrışma testi — EKG gömmeleri (HuBERT-ECG, 6 saatlik dilim)

**1)** **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU → Kaydet**

**2)** **Çalışma zamanı → Tümünü çalıştır** (uyarı çıkarsa *Yine de çalıştır*), Drive iznini ver.

Bu not defteri yalnızca gömme çıkarır (eğitim yok): 16 bitki × 71 pencere = 1136 pencere,
6 model (EKG + 5 rastgele başlangıç). Her modelin süresi ekrana yazılır. Sonuçlar Drive'a
(`bitkiEkg_sonuclar/18gun/`) kaydedilir; oturum koparsa **Tümünü çalıştır** yeter, bitenler atlanır.
Analiz yerelde yapılır (yöntem önceden yazıldı: `docs/on_kayit.md`). Bitince bana **"bitti"** yaz.

## 1. GPU kontrolü

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU yok! Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU, sonra Tümünü çalıştır.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

## 2. Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
KOK = Path('/content/drive/MyDrive/bitkiEkg_sonuclar/18gun')
KOK.mkdir(parents=True, exist_ok=True)
print('Sonuç klasörü:', KOK)

## 3. Kod ve veri

In [ ]:
!rm -rf /content/bitkiEkg
!git clone -q -b onDeneme --depth 1 https://github.com/melihakcam/bitkiEkg.git /content/bitkiEkg
!pip install -q "transformers>=4.40,<5" safetensors huggingface_hub
import sys
sys.path.insert(0, '/content/bitkiEkg/src')
import transformers
assert int(transformers.__version__.split('.')[0]) < 5, 'transformers 5 yüklü: Çalışma zamanı → Yeniden başlat, sonra Tümünü çalıştır.'
import time
import numpy as np
from bitki_ekg.dapt import hubert_govde
from bitki_ekg.sonda import hubert_gomme

z = np.load('/content/bitkiEkg/data/colab/domates_18gun_6h_500.npz')
X = np.tile(z['L'].astype(np.float32), (1, 12))
assert X.shape == (1136, 6000), X.shape
print('Veri:', X.shape, '| bitki:', len(np.unique(z['bitki'])), '| gün:', len(np.unique(z['gun'])))

## 4. Gömmeler: EKG ağırlıkları + 5 farklı rastgele başlangıç

In [ ]:
KOLLAR = [('ekg', True, 42)] + [(f'rastgele_t{t}', False, t) for t in [1, 2, 3, 4, 5]]
t_bas = time.time()
for i, (kol, onceden, tohum) in enumerate(KOLLAR, 1):
    dosya = KOK / f'gomme_{kol}_18gun.npz'
    if dosya.exists():
        print(f'[{i}/6] {kol}: önceden tamamlanmış, atlandı'); continue
    t0 = time.time()
    govde = hubert_govde(onceden, tohum)
    E = hubert_gomme(govde, X, 'cuda')
    np.savez_compressed(dosya, E=E, bitki=z['bitki'], gun=z['gun'], saat=z['saat'])
    del govde; torch.cuda.empty_cache()
    gecen = time.time() - t_bas
    print(f'[{i}/6] {kol}: {E.shape}, {time.time() - t0:.0f} s | toplam {gecen / 60:.1f} dk', flush=True)
print('Bitti. Dosyalar:', sorted(p.name for p in KOK.glob('gomme_*_18gun.npz')))